In [1]:
import time, io, ssl, certifi, urllib.request, requests
import pandas as pd, numpy as np
from datetime import datetime, timedelta
from contextlib import redirect_stdout

# -------------------------------------------------
# 0) Bring in your ESPN helpers (expects pull_players_and_teams_for_date)
# -------------------------------------------------
%run ESPNDATA.ipynb

def pull_players_and_teams_for_date_quiet(ds):
    buf = io.StringIO()
    with redirect_stdout(buf):
        return pull_players_and_teams_for_date(ds)

def norm_key(x):
    if pd.isna(x): return ""
    return str(x).strip().lower()

# -------------------------------------------------
# 1) (Optional but useful) Ensure team points exist in teams_df
#    Uses ESPN "summary" endpoint to fill missing points
# -------------------------------------------------
SITE_SUMMARY = "https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/summary"

def _j(url):
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    return r.json()

def _scores_for_event(event_id: str):
    h = _j(f"{SITE_SUMMARY}?event={event_id}").get("header", {})
    comp = (h.get("competitions") or [{}])[0]
    out = []
    for c in comp.get("competitors", []):
        team_name = c.get("team", {}).get("displayName")
        pts = c.get("score")
        if pts is None:
            ls = c.get("linescores") or []
            if ls:
                pts = sum(int(x.get("value") or 0) for x in ls)
        if team_name is not None and pts is not None:
            out.append({"event_id": str(event_id), "team": team_name, "points": int(pts)})
    return out

def ensure_points(teams_df: pd.DataFrame) -> pd.DataFrame:
    df = teams_df.copy()
    for candidate in ("points","pts","score"):
        if candidate in df.columns:
            return df.rename(columns={candidate: "points"}) if candidate != "points" else df

    score_rows = []
    for eid in df["event_id"].astype(str).unique():
        try:
            score_rows.extend(_scores_for_event(eid))
        except Exception:
            continue
        time.sleep(0.06)

    scores = pd.DataFrame(score_rows).dropna(subset=["points"])
    if scores.empty:
        raise ValueError("Couldn't derive team points; inspect teams_df and a sample summary JSON.")

    out = df.merge(scores, on=["event_id","team"], how="left")

    # fallback join by normalized team name if needed
    if out["points"].isna().any():
        df2 = out[out["points"].isna()].copy()
        ok  = out[out["points"].notna()]
        if not df2.empty:
            df2["team_key"] = df2["team"].map(norm_key)
            scores["team_key"] = scores["team"].map(norm_key)
            df2 = df2.drop(columns=["points"]).merge(
                scores.drop(columns=["team"]).drop_duplicates(["event_id","team_key"]),
                on=["event_id","team_key"], how="left"
            ).drop(columns=["team_key"])
            out = pd.concat([ok, df2], ignore_index=True)
    return out

# -------------------------------------------------
# 2) Pull raw team-level game rows for many dates (historical ingestion)
# -------------------------------------------------
def pull_team_totals_for_dates(date_list):
    rows = []
    for ds in date_list:
        _, teams_df = pull_players_and_teams_for_date_quiet(ds)
        if teams_df is None or teams_df.empty:
            continue
        rows.append(teams_df.assign(asof_date=ds))
        time.sleep(0.08)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

# -------------------------------------------------
# 3) Convert team rows -> one row per game (two teams)
# -------------------------------------------------
def build_game_table_from_teams(teams_df):
    t = ensure_points(teams_df.copy())
    t["team_key"] = t["team"].map(norm_key)

    g = t[["event_id","team","team_key","points","asof_date"]].copy()
    g = g.sort_values(["event_id","team_key"])

    pairs = []
    for eid, grp in g.groupby("event_id"):
        if len(grp) != 2:
            continue
        a, b = grp.iloc[0], grp.iloc[1]
        pairs.append({
            "event_id": str(eid),
            "team_a": a.team, "team_b": b.team,
            "team_a_key": a.team_key, "team_b_key": b.team_key,
            "pts_a": pd.to_numeric(a.points, errors="coerce"),
            "pts_b": pd.to_numeric(b.points, errors="coerce"),
            "asof_date": a.asof_date
        })

    games = pd.DataFrame(pairs).dropna(subset=["pts_a","pts_b"]).copy()
    games["total_points"] = games["pts_a"] + games["pts_b"]
    return games

# -------------------------------------------------
# 4) One-liner convenience: build historical games dataset
# -------------------------------------------------
def load_historical_games(n_days=200):
    today = datetime.utcnow().date()
    dates = [(today - timedelta(days=i)).strftime("%Y%m%d") for i in range(n_days)]
    team_rows = pull_team_totals_for_dates(dates)
    games = build_game_table_from_teams(team_rows)
    return games, team_rows

games, team_rows = load_historical_games(n_days=200)

print("team_rows:", team_rows.shape)
print("games:", games.shape)
games.head()


Found 16 games via: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?dates=2024&seasontype=2&week=1
Rows: 1228
 season  week             team stat_type           player athlete_id  seasontype  event_id C/ATT YDS  AVG  TD INT SACKS
   2024     1 Baltimore Ravens   passing    Lamar Jackson    3916387           2 401671789 26/41 273  6.7   1   0   1-6
   2024     1 Baltimore Ravens   rushing    Lamar Jackson    3916387           2 401671789   NaN 122  7.6   0 NaN   NaN
   2024     1 Baltimore Ravens   rushing    Derrick Henry    3043078           2 401671789   NaN  46  3.5   1 NaN   NaN
   2024     1 Baltimore Ravens   rushing      Zay Flowers    4429615           2 401671789   NaN  14  7.0   0 NaN   NaN
   2024     1 Baltimore Ravens   rushing     Justice Hill    4038441           2 401671789   NaN   3  3.0   0 NaN   NaN
   2024     1 Baltimore Ravens receiving    Isaiah Likely    4361050           2 401671789   NaN 111 12.3   1 NaN   NaN
   2024     1 Baltimore Rave

/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_1451/15705206.py:125: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().date()


team_rows: (544, 35)
games: (258, 9)


,event_id,team_a,team_b,team_a_key,team_b_key,pts_a,pts_b,asof_date,total_points
0,401772510,Dallas Cowboys,Philadelphia Eagles,dallas cowboys,philadelphia eagles,20.0,24.0,20250904,44.0
1,401772621,Chicago Bears,Philadelphia Eagles,chicago bears,philadelphia eagles,24.0,15.0,20251128,39.0
2,401772630,Green Bay Packers,Philadelphia Eagles,green bay packers,philadelphia eagles,7.0,10.0,20251110,17.0
3,401772631,Miami Dolphins,Washington Commanders,miami dolphins,washington commanders,16.0,13.0,20251116,29.0
4,401772632,Minnesota Vikings,Pittsburgh Steelers,minnesota vikings,pittsburgh steelers,21.0,24.0,20250928,45.0


In [ ]:
#Test model check if data is usable

import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

g = games.copy()
g["game_date"] = pd.to_datetime(g["asof_date"].astype(str), errors="coerce")
g = g.dropna(subset=["game_date","pts_a","pts_b","team_a_key","team_b_key"]).copy()

# Label: did team A win?
g["y_team_a_win"] = (g["pts_a"] > g["pts_b"]).astype(int)

# ---- build long format so we can compute rolling stats per team ----
a = g[["event_id","game_date","team_a_key","pts_a","pts_b"]].rename(
    columns={"team_a_key":"team_key","pts_a":"pts_for","pts_b":"pts_against"}
)
b = g[["event_id","game_date","team_b_key","pts_b","pts_a"]].rename(
    columns={"team_b_key":"team_key","pts_b":"pts_for","pts_a":"pts_against"}
)
long = pd.concat([a,b], ignore_index=True).sort_values(["team_key","game_date","event_id"])

# rolling means using ONLY games before current one (shift(1) prevents leakage)
window = 3
long["pf_l3"] = long.groupby("team_key")["pts_for"].shift(1).rolling(window).mean()
long["pa_l3"] = long.groupby("team_key")["pts_against"].shift(1).rolling(window).mean()

feats = long[["event_id","team_key","pf_l3","pa_l3"]].copy()

# merge features for team A and team B back into game rows
df = g.merge(feats, left_on=["event_id","team_a_key"], right_on=["event_id","team_key"], how="left") \
      .drop(columns=["team_key"]) \
      .rename(columns={"pf_l3":"a_pf_l3","pa_l3":"a_pa_l3"})

df = df.merge(feats, left_on=["event_id","team_b_key"], right_on=["event_id","team_key"], how="left") \
       .drop(columns=["team_key"]) \
       .rename(columns={"pf_l3":"b_pf_l3","pa_l3":"b_pa_l3"})

df = df.dropna(subset=["a_pf_l3","a_pa_l3","b_pf_l3","b_pa_l3"]).sort_values("game_date").reset_index(drop=True)

FEATURES = ["a_pf_l3","a_pa_l3","b_pf_l3","b_pa_l3"]
cut = int(len(df) * 0.8)

X_train, y_train = df.loc[:cut-1, FEATURES], df.loc[:cut-1, "y_team_a_win"].values
X_test,  y_test  = df.loc[cut:,   FEATURES], df.loc[cut:,   "y_team_a_win"].values

clf = LogisticRegression(solver="liblinear", max_iter=1000)
clf.fit(X_train, y_train)

p_test = clf.predict_proba(X_test)[:, 1]
y_hat = (p_test >= 0.5).astype(int)

print("Rows:", len(df), "| Train:", cut, "| Test:", len(df)-cut)
print("Accuracy:", round(accuracy_score(y_test, y_hat), 4))

preview = df.loc[cut:, ["game_date","team_a_key","team_b_key","pts_a","pts_b","y_team_a_win"]].copy()
preview["p_team_a_win"] = np.round(p_test, 3)
print(preview.head(10).to_string(index=False))


Rows: 209 | Train: 167 | Test: 42
Accuracy: 0.7143
 game_date         team_a_key           team_b_key  pts_a  pts_b  y_team_a_win  p_team_a_win
2025-11-23  arizona cardinals jacksonville jaguars   24.0   27.0             0         0.487
2025-11-23   baltimore ravens        new york jets   23.0   10.0             1         0.568
2025-11-23   los angeles rams tampa bay buccaneers   34.0    7.0             1         0.561
2025-11-23      chicago bears  pittsburgh steelers   31.0   28.0             1         0.507
2025-11-23 cincinnati bengals new england patriots   20.0   26.0             0         0.501
2025-11-23     dallas cowboys  philadelphia eagles   24.0   21.0             1         0.435
2025-11-23    atlanta falcons   new orleans saints   24.0   10.0             1         0.578
2025-11-23      detroit lions      new york giants   34.0   27.0             1         0.591
2025-11-23 indianapolis colts   kansas city chiefs   20.0   23.0             0         0.512
2025-11-23   seattl

In [ ]:
#builds data set and preps for model 

import pandas as pd, numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import LogisticRegression

# -----------------------------
# Helpers
# -----------------------------
def to_american_odds(p: float) -> float:
    """Convert win probability p to fair American odds (no vig)."""
    p = float(p)
    p = min(max(p, 1e-6), 1-1e-6)
    if p >= 0.5:
        return -100 * p / (1 - p)
    else:
        return 100 * (1 - p) / p

def _safe_numeric_cols(df, drop_like=()):
    """Pick numeric columns, excluding id-ish columns and obvious leakage columns."""
    if df is None or df.empty:
        return []
    num_cols = []
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in drop_like):
            continue
        if cl in {"event_id", "asof_date", "game_date", "team", "team_key", "home_away", "name", "athlete"}:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            num_cols.append(c)
    return num_cols

# -----------------------------
# 1) Pull ESPN data for many dates
# -----------------------------
def pull_espn_history(n_days=200):
    today = datetime.utcnow().date()
    dates = [(today - timedelta(days=i)).strftime("%Y%m%d") for i in range(n_days)]

    team_rows = []
    player_rows = []

    for ds in dates:
        players_df, teams_df = pull_players_and_teams_for_date_quiet(ds)
        if teams_df is not None and not teams_df.empty:
            team_rows.append(teams_df.assign(asof_date=ds))
        if players_df is not None and not players_df.empty:
            player_rows.append(players_df.assign(asof_date=ds))

    teams_all = pd.concat(team_rows, ignore_index=True) if team_rows else pd.DataFrame()
    players_all = pd.concat(player_rows, ignore_index=True) if player_rows else pd.DataFrame()
    return teams_all, players_all

# -----------------------------
# 2) Build one row per team per game with "all ESPN stats"
#    - team stats from teams_df
#    - summed player stats from players_df
# -----------------------------
def build_team_game_stats(teams_all: pd.DataFrame, players_all: pd.DataFrame) -> pd.DataFrame:
    if teams_all.empty:
        raise ValueError("teams_all is empty. ESPN pull returned no team data.")

    t = ensure_points(teams_all.copy())
    t["team_key"] = t["team"].map(norm_key)
    t["game_date"] = pd.to_datetime(t["asof_date"].astype(str), errors="coerce")

    # --- Team-level numeric stats (whatever ESPN gives you) ---
    # Drop obvious leakage columns *as raw stats* (we'll use rolled versions later if you want)
    leak_like_team = ("points", "pts", "score", "result", "winner", "win", "loss")
    team_num_cols = _safe_numeric_cols(t, drop_like=())  # keep everything numeric for now
    team_base = t[["event_id", "team_key", "team", "game_date", "points"] + team_num_cols].copy()

    # --- Player-level stats: sum numeric stats to team-game level ---
    if players_all is not None and not players_all.empty:
        p = players_all.copy()
        if "team" in p.columns and "team_key" not in p.columns:
            p["team_key"] = p["team"].map(norm_key)
        if "asof_date" in p.columns:
            p["game_date"] = pd.to_datetime(p["asof_date"].astype(str), errors="coerce")

        # Identify numeric player stat columns and sum them
        player_num_cols = _safe_numeric_cols(p, drop_like=("event_id","team"))
        keep_cols = [c for c in ["event_id","team_key"] if c in p.columns] + player_num_cols
        p_sums = (p[keep_cols]
                  .groupby(["event_id","team_key"], as_index=False)
                  .sum(numeric_only=True))

        # Prefix player stats so they don't collide with team stats
        rename_map = {c: f"p_{c}" for c in player_num_cols}
        p_sums = p_sums.rename(columns=rename_map)

        out = team_base.merge(p_sums, on=["event_id","team_key"], how="left")
    else:
        out = team_base

    return out


In [21]:
#actual model 
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, accuracy_score
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# -----------------------------
# 4) Train + create "your odds" (IMPUTE + FEATURE FILTER + CALIBRATE)
# -----------------------------
teams_all, players_all = pull_espn_history(n_days=200)
team_game = build_team_game_stats(teams_all, players_all)

# --- FIX: drop duplicate columns first (prevents df[col] being a DataFrame) ---
dup_cols = team_game.columns[team_game.columns.duplicated()].tolist()
print("Duplicate column names:", dup_cols[:30], ("..." if len(dup_cols) > 30 else ""))
print("Num duplicate columns:", len(dup_cols))

team_game = team_game.loc[:, ~team_game.columns.duplicated()].copy()

# --- now safely coerce numeric-looking columns ---
id_cols = {"event_id", "team_key", "team", "game_date"}

before_num = team_game.select_dtypes(include="number").columns.tolist()
print("Numeric cols BEFORE coercion:", len(before_num))

for c in team_game.columns:
    if c in id_cols:
        continue
    if team_game[c].dtype == "object":
        team_game[c] = pd.to_numeric(team_game[c], errors="coerce")

after_num = team_game.select_dtypes(include="number").columns.tolist()
print("Numeric cols AFTER coercion:", len(after_num))
print("Example numeric cols:", after_num[:30])

nonnull_rate = team_game[after_num].notna().mean().sort_values(ascending=False)
print("\nTop 20 numeric cols by non-null rate:")
print(nonnull_rate.head(20))


df, FEATURES = make_matchup_dataset(team_game, window=3)

# 1) Filter features by coverage so you don't end up with junk / 1 feature / tons of NaNs
coverage = df[FEATURES].notna().mean().sort_values(ascending=False)
FEATURES = coverage[coverage >= 0.25].index.tolist()   # keep features present in >=60% of rows

print("Features kept:", len(FEATURES))

# 2) Sort by time and make 3-way split (train / calibrate / test)
df = df.sort_values("game_date").reset_index(drop=True)
n = len(df)
cut_train = int(n * 0.70)
cut_cal   = int(n * 0.85)

train_df = df.iloc[:cut_train]
cal_df   = df.iloc[cut_train:cut_cal]
test_df  = df.iloc[cut_cal:]

# 3) Impute missing values (important for rolling stats)
imp = SimpleImputer(strategy="median")

X_train = imp.fit_transform(train_df[FEATURES])
y_train = train_df["y_team_a_win"].values

X_cal   = imp.transform(cal_df[FEATURES])
y_cal   = cal_df["y_team_a_win"].values

X_test  = imp.transform(test_df[FEATURES])
y_test  = test_df["y_team_a_win"].values

# 4) Fit logistic regression (linear baseline)
base = LogisticRegression(solver="liblinear", max_iter=4000, C=0.5)  # C lower = more regularization
base.fit(X_train, y_train)

# 5) Calibrate probabilities (THIS makes odds feel much more realistic)
cal = CalibratedClassifierCV(base, method="sigmoid", cv="prefit")
cal.fit(X_cal, y_cal)

# 6) Evaluate (log loss matters more for odds than accuracy)
p_test = cal.predict_proba(X_test)[:, 1]
y_hat = (p_test >= 0.5).astype(int)

print("Test accuracy:", round(accuracy_score(y_test, y_hat), 4))
print("Test logloss:", round(log_loss(y_test, p_test), 4))

# 7) Build odds table for the MOST RECENT games in test set
out = test_df[["game_date","team_a","team_b","pts_a","pts_b","y_team_a_win"]].copy()
out["p_team_a_win"] = np.round(p_test, 3)
out["fair_american_team_a"] = out["p_team_a_win"].apply(to_american_odds).round(1)
out["fair_american_team_b"] = (1 - out["p_team_a_win"]).apply(to_american_odds).round(1)

print("DF date range:", df["game_date"].min(), "→", df["game_date"].max())

latest_day = out["game_date"].max()
week_start = latest_day - pd.Timedelta(days=7)

print("\n=== LATEST WEEK (calibrated odds) ===")
print(out[out["game_date"] >= week_start].sort_values("game_date", ascending=False).head(30).to_string(index=False))


/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_1451/2537064857.py:36: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().date()


Duplicate column names: ['points'] 
Num duplicate columns: 1
Numeric cols BEFORE coercion: 1
Numeric cols AFTER coercion: 1
Example numeric cols: ['points']

Top 20 numeric cols by non-null rate:
points    0.99635
dtype: float64
Duplicate columns in tg: []
games shape: (274, 8)
pts_a dtype: float64
Example pts_a value type: <class 'numpy.float64'>
Example pts_a value: 20.0
Features kept: 1
Test accuracy: 0.5294
Test logloss: 0.7223
DF date range: 2025-09-04 00:00:00 → 2025-12-15 00:00:00

=== LATEST WEEK (calibrated odds) ===
 game_date               team_a                team_b  pts_a  pts_b  y_team_a_win  p_team_a_win  fair_american_team_a  fair_american_team_b
2025-12-15       Miami Dolphins   Pittsburgh Steelers   15.0   28.0             0         0.644                -180.9                 180.9
2025-12-14       Dallas Cowboys     Minnesota Vikings   26.0   34.0             0         0.806                -415.5                 415.5
2025-12-14        Chicago Bears      Cleveland B

/Users/rayanarya/Library/Python/3.12/lib/python/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [18]:
#model performance
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, brier_score_loss

# --- assumes you already have: test_df, y_test, p_test, out, FEATURES, and to_american_odds ---

print("\n=== ODDS QUALITY CHECKS ===")
print("Test logloss:", round(log_loss(y_test, p_test), 4))
print("Test brier:  ", round(brier_score_loss(y_test, p_test), 4))
print("p mean/std:  ", round(float(np.mean(p_test)), 4), "/", round(float(np.std(p_test)), 4))
print("p min/max:   ", round(float(np.min(p_test)), 4), "/", round(float(np.max(p_test)), 4))

# 1) Calibration table (reliability by probability bucket)
bins = np.linspace(0, 1, 11)  # 10 bins
tmp = pd.DataFrame({"p": p_test, "y": y_test})
tmp["bin"] = pd.cut(tmp["p"], bins=bins, include_lowest=True)

cal_table = (
    tmp.groupby("bin", observed=True)
       .agg(n=("y","size"),
            p_avg=("p","mean"),
            win_rate=("y","mean"))
       .reset_index()
)

# Expected Calibration Error (ECE)
cal_table["gap"] = (cal_table["p_avg"] - cal_table["win_rate"]).abs()
ece = (cal_table["n"] * cal_table["gap"]).sum() / cal_table["n"].sum()

print("\n=== CALIBRATION TABLE (10 bins) ===")
print(cal_table.to_string(index=False))
print("\nECE (lower is better):", round(float(ece), 4))

# 2) Show most extreme predictions (sanity check)
out2 = out.copy()
out2["p_team_a_win"] = p_test
out2["fair_american_team_a"] = out2["p_team_a_win"].apply(to_american_odds).round(0)
out2["fair_american_team_b"] = (1 - out2["p_team_a_win"]).apply(to_american_odds).round(0)

print("\n=== MOST CONFIDENT TEAM A PICKS ===")
print(out2.sort_values("p_team_a_win", ascending=False).head(10)[
    ["game_date","team_a","team_b","p_team_a_win","fair_american_team_a","y_team_a_win"]
].to_string(index=False))

print("\n=== MOST CONFIDENT TEAM B PICKS ===")
print(out2.sort_values("p_team_a_win", ascending=True).head(10)[
    ["game_date","team_a","team_b","p_team_a_win","fair_american_team_b","y_team_a_win"]
].to_string(index=False))

# 3) Feature sanity: are you accidentally running with only 1 feature?
print("\nFeatures used:", len(FEATURES))
print("Top 20 features:", FEATURES[:20])



=== ODDS QUALITY CHECKS ===
Test logloss: 0.7223
Test brier:   0.2627
p mean/std:   0.6158 / 0.101
p min/max:    0.3987 / 0.8059

=== CALIBRATION TABLE (10 bins) ===
       bin  n    p_avg  win_rate      gap
(0.3, 0.4]  2 0.398672  0.500000 0.101328
(0.4, 0.5]  3 0.480852  0.333333 0.147519
(0.5, 0.6] 10 0.558456  0.400000 0.158456
(0.6, 0.7] 11 0.653845  0.636364 0.017481
(0.7, 0.8]  7 0.730807  0.571429 0.159379
(0.8, 0.9]  1 0.805942  0.000000 0.805942

ECE (lower is better): 0.1278

=== MOST CONFIDENT TEAM A PICKS ===
 game_date               team_a                team_b  p_team_a_win  fair_american_team_a  y_team_a_win
2025-12-14       Dallas Cowboys     Minnesota Vikings      0.805942                -415.0             0
2025-12-14 Jacksonville Jaguars         New York Jets      0.781514                -358.0             1
2025-11-30 Jacksonville Jaguars      Tennessee Titans      0.754944                -308.0             1
2025-12-14  San Francisco 49ers      Tennessee Titans  